In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

In [2]:
def clean_insolvency(filepath, insolvency_type="company", out_csv=None):
    """
    Cleans and prepares UK Insolvency Service data for regression analysis.

    This function reads the national company insolvency statistics for England and Wales,
    converts the date and rate columns into appropriate formats, filters the data
    to a fixed time period (January 2019 to February 2025, however we start from year 2017
    to account for future analysis of lagged variables that may trim our data upto 2 years of data
    if we add lags 12 and 24) and exports a cleaned CSV for further analysis.

    Steps:
    1. Read the raw insolvency dataset from the given file path.
    2. Extract and rename key columns ("period" to "date", "tot_rate_ew" to "company_insolvency_rate_per_10k",
    "EW_individual_rate_per_10000" to "individual_insolvency_rate_per_10k").
    3. Convert date strings to datetime format and ensure the rate column is numeric.
    4. Sort records chronologically and limit them to the time period from January 2017 to February 2025.
    5. Save the cleaned dataset to a new CSV file.

    Parameters:
    filepath : str, optional
        Path to the raw Insolvency Service dataset.
    out_csv : str, optional
        Name of the output CSV file for cleaned data.

    Return:
    pandas.DataFrame
        Cleaned dataframe contains:
        - date (datetime64)
        - company_insolvency_rate_per_10k or individual_insolvency_rate_per_10k (float)
    """
    insolvency = pd.read_csv(filepath, header=1)

    if insolvency_type == "company":
        col_name = "tot_rate_ew"
        rename_col = "company_insolvency_rate_per_10k"
        default_out = "clean_company_insolvency_rates.csv"

    elif insolvency_type == "individual":
        col_name = "EW_individual_rate_per_10000"
        rename_col = "individual_insolvency_rate_per_10k"
        default_out = "clean_individual_insolvency_rates.csv"

    else:
        raise ValueError("insolvency_type must be either company or individual")

    out_csv = out_csv or default_out

    df = insolvency[["period", col_name]].copy()
    df["period"] = pd.to_datetime(df["period"], format="%Y-%m", errors="raise")
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")

    df = df.rename(columns={"period": "date", col_name: rename_col})
    df = df.sort_values("date", ascending=True)

    start = pd.Timestamp(2017, 1, 1)
    end = pd.Timestamp(2025, 2, 1)
    df = df[(df["date"] >= start) & (df["date"] <= end)].reset_index(drop=True)

    df.to_csv(out_csv, index=False)

    return df

In [3]:
def plot_comp_insolvency(rates_df):
    """
    Plots the England and Wales insolvency rate per 10000 companies
    with COVID, Brexit, and Energy Crisis shaded periods.

    The function visualises the cleaned insolvency dataset, showing how company liquidation rates changed
    over time. It also highlights the key external shocks that may have influenced insolvency trends,
    including COVID-19, the Brexit transition and the 2022 energy crisis.

    Parameters
    rates_df : pandas.DataFrame
        DataFrame containing at least:
        - date(datetime64)
        - insolvency_rate_per_10k(float)
    """

    plt.figure(figsize=(12, 6))
    plt.plot(rates_df["date"], rates_df["company_insolvency_rate_per_10k"],
             marker="o", linestyle="-")

    plt.title("England & Wales Insolvency Rate (per 10,000 companies)", fontsize=12)
    plt.xlabel("Date")
    plt.ylabel("Rate per 10,000 companies")

    plt.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2022-12-30"),
                color="red", alpha=0.1, label="COVID")
    plt.axvspan(pd.Timestamp("2020-01-31"), pd.Timestamp("2020-12-31"),
                color="blue", alpha=0.2, label="Brexit")
    plt.axvspan(pd.Timestamp("2021-10-01"), pd.Timestamp("2025-02-01"),
                color="orange", alpha=0.1, label="Energy crisis")

    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

In [4]:
def clean_macro(filepath, date_col, value_col, new_value_col_name, start="2017-01-01", end="2025-02-01", outfile=None, drop_missing=True, skiprows=None, names=None, frequency="M", calc_year_on_year=False):
    """
    Cleans and prepares macroeconomic data (such as inflation, GDP, or unemployment)
    for use as predictors in insolvency regression models.

    The function standardises and formats each economic indicator so it can be easily
    merged with the main insolvency dataset. It automatically detects date formats,
    converts them to a consistent monthly timeline, handles missing values, and can
    calculate year-on-year percentage changes if needed.

    In simpler terms, this function takes any raw ONS or similar macroeconomic dataset
    and turns it into a clean, ready-to-use time series covering the same analysis
    period as the insolvency data (January 2017 to February 2025).

    Steps:
    1. Reads the dataset (CSV or Excel) and keeps only the relevant date and value columns.
    2. Renames columns to standard names ('date' and the chosen indicator name).
    3. Automatically parses messy date formats and aligns all values to the first day of each month.
    4. For quarterly data, expands each quarter to three months for consistency.
    5. Optionally calculates year-on-year percentage change (e.g., inflation growth vs. last year).
    6. Filters the data to the fixed 2017–2025 range.
    7. Removes or fills any missing values.
    8. Optionally saves and downloads the cleaned file.

    Parameters:
    filepath : str
        Path to the input dataset (CSV or Excel).
    date_col : str
        The column that contains dates or quarters.
    value_col : str
        The column containing the economic value.
    new_value_col_name : str
        The name to give this variable in the cleaned dataset.
    start, end : str, optional
        Date range to keep (default January 2017 to February 2025).
    outfile : str, optional
        Saves and downloads the cleaned dataset.
    drop_missing : bool, optional
        If True (default), drops rows with missing values.
    frequency : {"M", "Q"}, optional
        Data frequency - monthly (default) or quarterly.
    calc_year_on_year : bool, optional
        If True, computes year-on-year % change in numbers.

    Return:
    pandas.DataFrame
        Cleaned macroeconomic dataset contains:
        - date(datetime)
        - new_value_col_name(float)

    """
    df=pd.read_csv(filepath, skiprows=skiprows, names=names) if filepath.endswith(".csv") else pd.read_excel(filepath, skiprows=skiprows, names=names)
    df = df[[date_col, value_col]].copy()
    df=df.rename(columns={date_col:"date", value_col:new_value_col_name})
    df[new_value_col_name]=pd.to_numeric(df[new_value_col_name], errors="coerce")

    if frequency=="M":
      s = df["date"].astype(str)

      format_candidates = ["%b %Y", "%Y %b", "%Y-%m", "%m/%Y", "%Y/%m"]
      parsed = None
      best_nonnull = -1
      for fmt in format_candidates:
        try_dt = pd.to_datetime(s, format=fmt, errors="coerce")
        nn = try_dt.notna().sum()
        if nn > best_nonnull:
          best_nonnull = nn
          parsed = try_dt
        if nn == len(s):
          break

      if parsed is None or best_nonnull == 0:
        parsed = pd.to_datetime(s, errors="coerce")

      df["date"] = parsed
      df = df.dropna(subset=["date"])
      df["date"] = df["date"] + pd.offsets.MonthBegin(0)

      if calc_year_on_year:
        df[new_value_col_name] = df[new_value_col_name].pct_change(periods=12) * 100
        df = df[df["date"] >= "2017-01-01"]
    elif frequency=="Q":

      if calc_year_on_year:
        df[new_value_col_name] = df[new_value_col_name].pct_change(periods=4) * 100

      out_rows = []
      for _, row in df.iterrows():
          try:
              txt = str(row["date"]).strip()
              parts = txt.split()
              if parts[0].upper().startswith("Q"):   # if Q3 2010
                  q = int(parts[0][1:]); year = int(parts[1])
              else:                                   # if 2010 Q3
                  year = int(parts[0]); q = int(parts[1][1:])
          except Exception:
              continue

          if q == 1:
              months = [1, 2, 3]
          elif q == 2:
              months = [4, 5, 6]
          elif q == 3:
              months = [7, 8, 9]
          else:
              months = [10, 11, 12]

          for m in months:
              out_rows.append({
                  "date": pd.Timestamp(year=year, month=m, day=1),
                  new_value_col_name: row[new_value_col_name]
              })
      df = pd.DataFrame(out_rows)
      if calc_year_on_year and not df.empty:
          df = df[df["date"] >= "2017-01-01"]

    df=df[(df["date"]>=start) & (df["date"]<=end)]
    df=df.sort_values("date", ascending=True)
    df=df.reset_index(drop=True)

    if drop_missing:
      df=df.dropna(subset=[new_value_col_name])
    else:
      df[new_value_col_name]=df[new_value_col_name].fillna(df[new_value_col_name].mean())

    if outfile:
      df.to_csv(outfile, index=False)
      files.download(outfile)
    return df

In [5]:
def plot_macro(df, value_col, title, ylabel):
    """
    Plots a time series of a macroeconomic indicator over time.

    Creates a simple line plot showing the evolution of a chosen economic variable
    against time. This function is used to visualise trends in cleaned predictor
    data before running regression analysis.

    Parameters:
    df : pandas.DataFrame
        DataFrame containing a 'date' column and at least one numeric variable.
    value_col : str
        Name of the column to plot on the y-axis.
    title : str
        Plot title describing the indicator.
    ylabel : str
        Label for the y-axis.
    """
    plt.figure(figsize=(12,6))
    plt.plot(df["date"], df[value_col], marker="o", linestyle="-")

    plt.title(title, fontsize=12)
    plt.xlabel("Date")
    plt.ylabel(ylabel)

    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()